In [ ]:
from google.colab import drive
drive.mount('/content/drive') 

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

dataset_path = '/kaggle/input/skin-cancer-mnist-ham10000'
csv_path = os.path.join(dataset_path, 'HAM10000_metadata.csv')

df = pd.read_csv(csv_path)

def caminho_da_imagem(image_id):
    nome_arquivo = f"{image_id}.jpg"
    
    caminho_part1 = os.path.join(dataset_path, 'ham10000_images_part_1', nome_arquivo)
    if os.path.exists(caminho_part1):
        return caminho_part1
        
    return os.path.join(dataset_path, 'ham10000_images_part_2', nome_arquivo)

df['image_path'] = df['image_id'].apply(caminho_da_imagem)

df['label_idx'] = pd.Categorical(df['dx']).codes
train_df, val_df = train_test_split(df, test_size=0.4, random_state=42, stratify=df['label_idx'])


val_df, test_df = train_test_split(val_df, test_size=0.4, random_state=42, stratify=val_df['label_idx'])


In [ ]:
import tensorflow as tf

def carregar_e_processar_classificacao(caminho_img, label):

    img_raw = tf.io.read_file(caminho_img)
    img = tf.image.decode_jpeg(img_raw, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    
    return img, label

batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices((train_df['image_path'].values, train_df['label_idx'].values))
train_dataset = train_dataset.map(carregar_e_processar_classificacao, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_df['image_path'].values, val_df['label_idx'].values))
val_dataset = val_dataset.map(carregar_e_processar_classificacao, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)


In [ ]:
import matplotlib.pyplot as plt

for imagens_lote, labels_lote in train_dataset.take(1):
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
    for i in range(3):
        
        axes[i].imshow(imagens_lote[i].numpy())
        axes[i].set_title(f"Classe ID: {labels_lote[i].numpy()}")
        axes[i].axis('off')
    plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)


base_model.trainable = False


model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), 
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4), 
    layers.Dense(7, activation='softmax') 
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs_range = range(len(acc))

plt.figure(figsize=(6, 4))
plt.plot(epochs_range, acc, label='Treino (60%)')
plt.plot(epochs_range, val_acc, label='Validação (25%)')
plt.title('Progresso do Modelo')
plt.legend()
plt.show()

In [ ]:

train_df = pd.concat([train_df, val_df], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

train_dataset = tf.data.Dataset.from_tensor_slices((train_df['image_path'].values, train_df['label_idx'].values))
train_dataset = train_dataset.map(carregar_e_processar_classificacao, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((test_df['image_path'].values, test_df['label_idx'].values))
test_dataset = test_dataset.map(carregar_e_processar_classificacao, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(buffer_size=tf.data.AUTOTUNE)


model.fit(train_dataset, epochs=3)

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([y for x, y in test_dataset], axis=0)
predictions = model.predict(test_dataset)
y_pred = np.argmax(predictions, axis=1)

classes_nomes = pd.Categorical(df['dx']).categories.tolist()
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=classes_nomes, yticklabels=classes_nomes)
plt.title('Matriz de Confusão Final (Dados de Teste - 15%)')
plt.show()

print(classification_report(y_true, y_pred, target_names=classes_nomes))